In [2]:
"""
================================================================================
SELECCIÓN DE 200 TWEETS PARA GOLD STANDARD
Muestreo estratificado por: temporalidad, clase y confianza del modelo
================================================================================
"""

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# 1. CARGAR ARCHIVO
# ============================================================
print("Sube tu archivo Excel con los tweets clasificados:")
uploaded = files.upload()
archivo = list(uploaded.keys())[0]

df = pd.read_excel(archivo)
print(f"\n✓ {len(df):,} tweets cargados")

# ============================================================
# 2. PREPARAR DATOS
# ============================================================
df['TweetCreateTime'] = pd.to_datetime(df['TweetCreateTime'])
df['Fecha'] = df['TweetCreateTime'].dt.date

# Convertir confianza (puede venir con coma decimal)
df['confianza_sentimiento'] = (
    df['confianza_sentimiento']
    .astype(str)
    .str.replace(',', '.')
    .astype(float)
)

# ============================================================
# 3. CREAR ESTRATOS
# ============================================================

# Estrato temporal: 3 bloques
df['bloque_temporal'] = pd.cut(
    df['TweetCreateTime'].astype(np.int64),
    bins=3,
    labels=['Abril_1ra_quincena', 'Abril_2da_quincena', 'Mayo']
)

# Estrato de clase: la etiqueta del modelo
df['clase'] = df['sentimiento_numerico'].map({
    -1: 'PRO_CASTILLO',
     0: 'NEUTRAL',
     1: 'PRO_KEIKO'
})

# Estrato de confianza: 3 niveles
def nivel_confianza(c):
    if c >= 0.80:
        return 'alta'
    elif c >= 0.50:
        return 'media'
    else:
        return 'baja'

df['nivel_confianza'] = df['confianza_sentimiento'].apply(nivel_confianza)

# ============================================================
# 4. MOSTRAR DISTRIBUCIÓN DEL CORPUS
# ============================================================
print(f"\n{'='*60}")
print("DISTRIBUCIÓN DEL CORPUS COMPLETO")
print(f"{'='*60}")

print("\nPor clase:")
dist_clase = df['clase'].value_counts(normalize=True)
for clase, pct in dist_clase.items():
    print(f"  {clase}: {pct*100:.1f}%")

print("\nPor bloque temporal:")
dist_tiempo = df['bloque_temporal'].value_counts(normalize=True)
for bloque, pct in dist_tiempo.items():
    print(f"  {bloque}: {pct*100:.1f}%")

print("\nPor nivel de confianza:")
dist_conf = df['nivel_confianza'].value_counts(normalize=True)
for nivel, pct in dist_conf.items():
    print(f"  {nivel}: {pct*100:.1f}%")

# ============================================================
# 5. SELECCIONAR 200 TWEETS ESTRATIFICADOS
# ============================================================
N_TOTAL = 200
np.random.seed(42)  # Reproducibilidad

# Calcular cuántos tweets por cada combinación de estratos
# proporcional a su frecuencia en el corpus
df['estrato'] = df['clase'] + '_' + df['nivel_confianza'] + '_' + df['bloque_temporal'].astype(str)

frecuencias = df['estrato'].value_counts(normalize=True)
cuotas = (frecuencias * N_TOTAL).round().astype(int)

# Ajustar para que sumen exactamente 200
diferencia = N_TOTAL - cuotas.sum()
if diferencia > 0:
    # Agregar al estrato más grande
    cuotas.iloc[0] += diferencia
elif diferencia < 0:
    # Quitar del estrato más grande
    cuotas.iloc[0] += diferencia

# Seleccionar tweets de cada estrato
seleccionados = []
for estrato, n in cuotas.items():
    pool = df[df['estrato'] == estrato]
    if len(pool) == 0:
        continue
    n_real = min(n, len(pool))
    if n_real > 0:
        muestra = pool.sample(n=n_real, random_state=42)
        seleccionados.append(muestra)

gold = pd.concat(seleccionados, ignore_index=True)

# Si faltan tweets para llegar a 200 (por estratos muy pequeños),
# completar con muestreo aleatorio del resto
if len(gold) < N_TOTAL:
    ids_ya = set(gold['ID'].values)
    restantes = df[~df['ID'].isin(ids_ya)]
    faltantes = N_TOTAL - len(gold)
    extra = restantes.sample(n=faltantes, random_state=42)
    gold = pd.concat([gold, extra], ignore_index=True)

# Recortar si hay más de 200
gold = gold.head(N_TOTAL)

# Ordenar por fecha
gold = gold.sort_values('TweetCreateTime').reset_index(drop=True)

# ============================================================
# 6. CREAR ARCHIVO PARA ETIQUETAR
# ============================================================

# Solo las columnas necesarias + columna vacía para etiqueta humana
gold_output = gold[[
    'ID', 'TweetCreateTime', 'Handle', 'FollowersCount',
    'TweetText', 'tweet_homologado',
    'sentimiento_economico', 'sentimiento_numerico',
    'confianza_sentimiento', 'nivel_confianza', 'bloque_temporal'
]].copy()

# Columna vacía para que llenen manualmente
gold_output['ETIQUETA_HUMANA'] = ''

# ============================================================
# 7. MOSTRAR DISTRIBUCIÓN DE LA MUESTRA
# ============================================================
print(f"\n{'='*60}")
print(f"GOLD STANDARD: {len(gold_output)} TWEETS SELECCIONADOS")
print(f"{'='*60}")

print("\nPor clase del modelo:")
dist = gold['clase'].value_counts()
for clase, n in dist.items():
    pct = n / len(gold) * 100
    print(f"  {clase}: {n} ({pct:.1f}%)")

print("\nPor nivel de confianza:")
dist_c = gold['nivel_confianza'].value_counts()
for nivel, n in dist_c.items():
    pct = n / len(gold) * 100
    print(f"  {nivel}: {n} ({pct:.1f}%)")

print("\nPor bloque temporal:")
dist_t = gold['bloque_temporal'].value_counts()
for bloque, n in dist_t.items():
    pct = n / len(gold) * 100
    print(f"  {bloque}: {n} ({pct:.1f}%)")

print(f"\nRango de fechas: {gold['Fecha'].min()} a {gold['Fecha'].max()}")

# ============================================================
# 8. GUARDAR Y DESCARGAR
# ============================================================
archivo_salida = 'GOLD_STANDARD_200_tweets_para_etiquetar.xlsx'
gold_output.to_excel(archivo_salida, index=False)

print(f"\n{'='*60}")
print("INSTRUCCIONES DE ETIQUETADO")
print(f"{'='*60}")
print("""
1. Abrir el archivo descargado en Excel/Google Sheets
2. Leer la columna 'TweetText' (el tweet original)
3. En la columna 'ETIQUETA_HUMANA' poner:

   +1  = PRO_KEIKO / pro-mercado / anti-Castillo
   -1  = PRO_CASTILLO / anti-Keiko / incertidumbre
    0  = NEUTRAL / informativo / ambiguo

4. IMPORTANTE: Clasificar por la INTENCIÓN REAL del tweet.
   Si hay sarcasmo, clasificar por lo que realmente quiere decir,
   no por las palabras literales.

5. IMPORTANTE: Etiquetar de forma INDEPENDIENTE (cada investigador
   por separado, sin ver las etiquetas del otro).

6. Las columnas 'sentimiento_economico' y 'sentimiento_numerico'
   muestran lo que dijo el modelo. Están ahí como referencia
   pero NO deben influir en la etiqueta humana.

7. Cuando ambos terminen, comparar y resolver desacuerdos.
""")

files.download(archivo_salida)
print(f"✓ Descargando: {archivo_salida}")

Sube tu archivo Excel con los tweets clasificados:


Saving tweets_limpios (3)_V3_BALANCEADO (2).xlsx to tweets_limpios (3)_V3_BALANCEADO (2).xlsx

✓ 97,118 tweets cargados

DISTRIBUCIÓN DEL CORPUS COMPLETO

Por clase:
  PRO_CASTILLO: 43.6%
  NEUTRAL: 32.5%
  PRO_KEIKO: 23.9%

Por bloque temporal:
  Mayo: 39.6%
  Abril_1ra_quincena: 31.8%
  Abril_2da_quincena: 28.7%

Por nivel de confianza:
  alta: 67.7%
  baja: 29.4%
  media: 2.9%

GOLD STANDARD: 200 TWEETS SELECCIONADOS

Por clase del modelo:
  PRO_CASTILLO: 87 (43.5%)
  NEUTRAL: 64 (32.0%)
  PRO_KEIKO: 49 (24.5%)

Por nivel de confianza:
  alta: 134 (67.0%)
  baja: 59 (29.5%)
  media: 7 (3.5%)

Por bloque temporal:
  Mayo: 79 (39.5%)
  Abril_1ra_quincena: 64 (32.0%)
  Abril_2da_quincena: 57 (28.5%)

Rango de fechas: 2021-04-02 a 2021-05-31

INSTRUCCIONES DE ETIQUETADO

1. Abrir el archivo descargado en Excel/Google Sheets
2. Leer la columna 'TweetText' (el tweet original)
3. En la columna 'ETIQUETA_HUMANA' poner:

   +1  = PRO_KEIKO / pro-mercado / anti-Castillo
   -1  = PRO_CASTILLO 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Descargando: GOLD_STANDARD_200_tweets_para_etiquetar.xlsx
